# Penalty and barrier methods

Both turn a constrained problem into a *sequence* of unconstrained ones and
hand each to an ordinary unconstrained solver. They differ in where the
iterates live:

- `PenaltyMethod` charges for violation, so iterates approach the feasible set from **outside** as $\tau \to \infty$
- `PenaltyBarrierMethod` charges for nearing the boundary, so iterates stay strictly **inside** as $\tau \to 0$

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

from mopt.nonlinear import (ConstrainedNLPProblem, PenaltyMethod, PenaltyBarrierMethod,
                            TrustRegion, QuasiNewton, ConjugateGradient,
                            dogleg, constraint_violation, penalty_merit, barrier_merit)

## Test problem

$$\min_x\; x^T Q x + b^T x \quad\text{s.t.}\quad c^T x = -10,\;\; x^T M x \le 82$$

with $Q$ and $M$ symmetric positive definite. The constraints are given
algebraically (`ineq`/`eq`) because these methods fold them into a merit
function — unlike the oracle-based solvers.

In [2]:
Q = np.array([
    [5.9, -0.1, 0.1, 0.3, -1.1],
    [-0.1, 7.9, -0.2, -0.3, -0.1],
    [0.1, -0.2, 6.9, 0.1, -0.3],
    [0.3, -0.3, 0.1, 8.4, 0.1],
    [-1.1, -0.1, -0.3, 0.1, 6.8],
])
M = np.array([
    [5.8, -1.3, -1.2, 0.2, 1.2],
    [-1.3, 3.5, -1.9, -1.0, -1.8],
    [-1.2, -1.9, 4.4, 1.3, 1.3],
    [0.2, -1.0, 1.3, 4.1, -2.0],
    [1.2, -1.8, 1.3, -2.0, 6.1],
])
b = np.array([3.0, -3.0, -2.0, -2.0, 1.0])
c = np.array([-4.0, -1.0, 5.0, 4.0, 1.0])

f = lambda x: float(x @ Q @ x + b @ x)

def make_problem(x0):
    return ConstrainedNLPProblem(
        f=f, x0=x0,
        grad=lambda x: 2.0 * (Q @ x) + b,
        hess=lambda x: 2.0 * Q,
        ineq=lambda x: np.array([x @ M @ x - 82.0]),
        ineq_jac=lambda x: (2.0 * (M @ x))[None, :],
        eq=lambda x: np.array([c @ x + 10.0]),
        eq_jac=lambda x: c[None, :],
    )

problem = make_problem(np.zeros(5))
reference = minimize(f, np.zeros(5), jac=problem.grad, method="SLSQP", tol=1e-12,
                     constraints=[{"type": "eq", "fun": problem.eq},
                                  {"type": "ineq", "fun": lambda z: -problem.ineq(z)}])
print(f"scipy SLSQP reference: f = {reference.fun:.9f}")
print(f"x = {np.round(reference.x, 6)}")

scipy SLSQP reference: f = 16.032517310
x = [ 0.716032  0.331929 -0.879183 -0.554238 -0.191081]


## Both methods, across inner solvers and schedules

`inner_solver` is what minimizes each merit function; `tau0` and `theta` set
the schedule. Anything satisfying `BaseOptimizer` can go in the slot.

In [3]:
inner_solvers = [
    ("TrustRegion(dogleg)", lambda: TrustRegion(method=dogleg)),
    ("QuasiNewton", lambda: QuasiNewton()),
    ("ConjugateGradient", lambda: ConjugateGradient()),
]

rows = []
for tau0 in [0.1, 1.0, 10.0]:
    for theta in [0.1, 0.5]:
        for name, make_inner in inner_solvers:
            pen = PenaltyMethod(inner_solver=make_inner(), tau0=tau0, theta=theta)
            r_p = pen.solve(make_problem(np.zeros(5)))
            bar = PenaltyBarrierMethod(inner_solver=make_inner(), tau0=tau0, theta=theta)
            r_b = bar.solve(make_problem(np.zeros(5)))
            rows.append({
                "tau0": tau0, "theta": theta, "inner": name,
                ("penalty", "ok"): r_p.success,
                ("penalty", "outer"): r_p.n_iter,
                ("penalty", "f"): r_p.fun,
                ("penalty", "violation"): constraint_violation(problem, r_p.x),
                ("barrier", "ok"): r_b.success,
                ("barrier", "outer"): r_b.n_iter,
                ("barrier", "f"): r_b.fun,
                ("barrier", "violation"): constraint_violation(problem, r_b.x),
            })

table = pd.DataFrame(rows).set_index(["tau0", "theta", "inner"])
table.columns = pd.MultiIndex.from_tuples(table.columns)
table

penalty                                barrier  \
                                    ok outer          f     violation      ok   
tau0 theta inner                                                                
0.1  0.1   TrustRegion(dogleg)    True     9  16.032517  1.411848e-07    True   
           QuasiNewton           False     9  16.032517  1.411849e-07   False   
           ConjugateGradient     False     9  16.032119  1.411841e-04   False   
     0.5   TrustRegion(dogleg)    True    25  16.032515  8.415271e-07    True   
           QuasiNewton           False    28  16.032517  1.411849e-07   False   
           ConjugateGradient     False    28  16.031909  2.154272e-04   False   
1.0  0.1   TrustRegion(dogleg)    True     8  16.032517  1.411848e-07    True   
           QuasiNewton           False     8  16.032517  1.411849e-07   False   
           ConjugateGradient     False     8  16.032119  1.411832e-04   False   
     0.5   TrustRegion(dogleg)    True    22  16.032515  6.732217e-07    True   
           QuasiNewton           False    25  16.032517  1.411849e-07   False   
           ConjugateGradient     False    25  16.032031  1.723427e-04   False   
10.0 0.1   TrustRegion(dogleg)    True     7  16.032517  1.411848e-07    True   
           QuasiNewton           False     7  16.032517  1.411849e-07   False   
           ConjugateGradient     False     7  16.032119  1.411861e-04   False   
     0.5   TrustRegion(dogleg)    True    19  16.032516  5.385774e-07    True   
           QuasiNewton           False    21  16.032517  1.411849e-07   False   
           ConjugateGradient     False    21  16.032128  1.378746e-04   False   

                                                               
                               outer          f     violation  
tau0 theta inner                                               
0.1  0.1   TrustRegion(dogleg)     8  16.032517  1.411848e-07  
           QuasiNewton             8  16.032517  1.411848e-07  
           ConjugateGradient       8  16.032119  1.411833e-04  
     0.5   TrustRegion(dogleg)    21  16.032517  1.411848e-07  
           QuasiNewton            21  16.032517  1.411849e-07  
           ConjugateGradient      21  16.032128  1.378748e-04  
1.0  0.1   TrustRegion(dogleg)     9  16.032517  1.411848e-07  
           QuasiNewton             9  16.032517  1.411846e-07  
           ConjugateGradient       9  16.032119  1.411834e-04  
     0.5   TrustRegion(dogleg)    25  16.032517  1.411848e-07  
           QuasiNewton            25  16.032517  1.411848e-07  
           ConjugateGradient      25  16.032031  1.723425e-04  
10.0 0.1   TrustRegion(dogleg)    10  16.032517  1.411848e-07  
           QuasiNewton            10  16.032517  1.411851e-07  
           ConjugateGradient      10  16.032119  1.411830e-04  
     0.5   TrustRegion(dogleg)    28  16.032517  1.411848e-07  
           QuasiNewton            28  16.032517  1.411849e-07  
           ConjugateGradient      28  16.031909  2.154275e-04

Two things to read off this table.

`ok` is not just feasibility. Both methods require the *final inner
minimization* to have converged as well, because feasibility alone proves
nothing — the barrier keeps every iterate feasible by construction, so a
stalled run would otherwise report success. A weaker inner solver can
therefore land on essentially the right answer and still report `False`; the
reason is in `result.message`.

`TrustRegion(dogleg)` is the default for a reason. The merit function grows
ill-conditioned as the schedule advances — its Hessian carries $\tau$ in the
constraint directions — and a trust region copes with that far better than the
line-search methods do.

## Where the iterates live

The defining difference. Run the penalty method with a capped $\tau$ and look
at the inequality: it is satisfied only in the limit, whereas the barrier
method is strictly inside at every $\tau$.

In [4]:
print(f"{'tau_max':>10} {'g(x) = xMx - 82':>18} {'|h(x)|':>12}   penalty method")
for tau_max in [1e1, 1e3, 1e5, 1e7]:
    r = PenaltyMethod(tau_max=tau_max).solve(make_problem(np.zeros(5)))
    g = float(r.x @ M @ r.x - 82.0)
    h = abs(float(c @ r.x + 10.0))
    print(f"{tau_max:>10.0e} {g:>18.6f} {h:>12.2e}")

print()
print(f"{'tau_min':>10} {'g(x) = xMx - 82':>18} {'|h(x)|':>12}   barrier method")
for tau_min in [1e-1, 1e-3, 1e-5, 1e-7]:
    r = PenaltyBarrierMethod(tau_min=tau_min).solve(make_problem(np.zeros(5)))
    g = float(r.x @ M @ r.x - 82.0)
    h = abs(float(c @ r.x + 10.0))
    print(f"{tau_min:>10.0e} {g:>18.6f} {h:>12.2e}")

   tau_max    g(x) = xMx - 82       |h(x)|   penalty method
     1e+01         -70.678039     1.40e-01
     1e+03         -70.369606     1.41e-03
     1e+05         -70.366463     1.41e-05
     1e+07         -70.366432     1.41e-07

   tau_min    g(x) = xMx - 82       |h(x)|   barrier method
     1e-01         -70.680165     1.40e-01
     1e-03         -70.369624     1.41e-03
     1e-05         -70.366464     1.41e-05
     1e-07         -70.366432     1.41e-07


The equality violation is what shrinks in both cases; the inequality is
inactive at this optimum, so the barrier simply keeps it strictly negative
throughout.

## The merit functions are ordinary problems

`penalty_merit` and `barrier_merit` return an `NLPProblem`, which is exactly
why any unconstrained solver can be dropped into `inner_solver`. You can
inspect one directly.

In [5]:
merit = penalty_merit(problem, tau=10.0)
print(type(merit).__name__)
x = np.array([0.5, 0.0, -0.5, 0.0, 0.0])
print(f"f(x)            = {f(x):.6f}")
print(f"merit(x, tau=10) = {merit.f(x):.6f}   (f plus the penalty on violation)")
print(f"gradient shape   = {merit.grad(x).shape},  Hessian shape = {merit.hess(x).shape}")

NLPProblem
f(x)            = 5.650000
merit(x, tau=10) = 308.150000   (f plus the penalty on violation)
gradient shape   = (5,),  Hessian shape = (5, 5)


## Requirements are checked

The barrier needs a strictly feasible start, since its merit function is
infinite outside the feasible set.

In [6]:
outside = np.array([4.0, 0.0, 0.0, 0.0, 0.0])
print("g(x0) =", float(outside @ M @ outside - 82.0), "> 0, so x0 is infeasible")
try:
    PenaltyBarrierMethod().solve(make_problem(outside))
except ValueError as exc:
    print("barrier ->", exc)

r = PenaltyMethod().solve(make_problem(outside))   # penalty does not care
print(f"penalty -> success={r.success}, f={r.fun:.9f}")

g(x0) = 10.799999999999997 > 0, so x0 is infeasible
barrier -> PenaltyBarrierMethod needs a strictly feasible x0: max(ineq(x0)) = 10.8, must be < 0.
penalty -> success=True, f=16.032516911
